# 03 · Эксперименты NBCO

**Пайплайн НИР — стадия 3 из 3.** Улучшение существующей сети через
NBCO (learned construction + trim/extend edit-политика). Основные результаты статьи:

- **единичный прогон** (dry-run плана + один запуск);
- **alpha-sweep** (E1: neural BCO vs Our NBCO по компромиссу RTT↔WMC);
- **E2 Pareto** (alpha × adj_target);
- **ablation** с разным набором пчёл (E2 5-model);
- **MACSA Table B**, **EKB case study**.

Всё параметризуется в коде через `build_experiment(...)` / `run_experiment(...)`
(M015): город, alpha, adj_target, число итераций, границы маршрута — аргументами,
`None` ⇒ значение из YAML. Логика — в `search/` и `paper_experiments/`.

In [ ]:
import pandas as pd
from IPython.display import display

from connectpt.routes_generator.core import (
    load_suite, build_experiment, ExperimentRunFactory)
from connectpt.routes_generator.paper_experiments.paper_runs import (
    run_experiment, run_batch, save_paper)
from connectpt.routes_generator import render_report

## Профиль запуска

`suite_smoke` — быстрый прогон (2 итерации, префикс `TEMP_`); `suite` — полный
бюджет статьи. Профиль задаёт только бюджет/пути; выбор эксперимента — вызовами
ниже.

In [ ]:
PROFILE = 'smoke'     # 'smoke' | 'full'
suite = load_suite('suite_smoke' if PROFILE == 'smoke' else 'suite')
print('smoke:', suite.smoke, '| префикс:', repr(suite.output_prefix))

## Единичный прогон

Сначала — **dry-run плана** (состав пчёл + какие модели грузятся, без BCO-цикла),
затем один запуск Our NBCO на Mandl при фиксированном alpha.

In [ ]:
cfg = build_experiment('e1/our_nbco', city='Mandl', alpha=0.5, smoke=suite.smoke)
plan = ExperimentRunFactory.from_cfg(cfg).run(dry_run=True)
print('модели   :', plan.metadata['models_loaded'])
print('пчёлы    :', plan.plan['counts'])

single = run_experiment('e1/our_nbco', suite, city='Mandl', alpha=0.5)
single.display()

## Alpha-sweep (E1): neural BCO vs Our NBCO

`run_batch("e1/batch", suite, city=...)` гоняет оба метода по сетке alpha
(компромисс время-в-пути ↔ связность) и строит сравнительную таблицу + Pareto.

In [ ]:
e1 = run_batch('e1/batch', suite, city='Mandl')
e1.display()

## E2 · Pareto-фронт (alpha × adj_target)

Двумерный свип Our NBCO на Mumford1: компромисс качества и степени изменения сети
(`adj_target`).

In [ ]:
e2 = run_experiment('e2/our_pareto/mumford1/our_nbco', suite)
e2.display()

## Ablation — разный набор пчёл (E2 5-model)

Пять вариантов пчелиного состава (GNN/RPC × trim-extend/type2 + trim12+extend12)
на Mumford1, adjustment выключен. Набор пчёл и модели приходят из групп
`search/bee_sets/*` + `search/models/*` — один батч, разные листья.

In [ ]:
e2_5 = run_batch('e2/5model/mumford1/batch', suite)
e2_5.display()

## MACSA · Table B

Скоринг фиксированных маршрутов Table-B + alpha-sweep Our NBCO + выбор лучшего
решения, побеждающего MACSA. Один вызов (в smoke — вариант iter=1 из статьи).

In [ ]:
from connectpt.routes_generator.paper_experiments.macsa_run import run_macsa_table_b

macsa = run_macsa_table_b(suite)
macsa.display()

## EKB · case study

Реальная сеть Екатеринбурга: прогон Our NBCO (GNN rebuild + trim/extend) с гео-рендером.

In [ ]:
ekb = run_experiment('ekb/case_study/our_nbco', suite, kind='gis', title='EKB NBCO')
ekb.display()

## Итог

Все таблицы/фигуры сохранены в папку результатов профиля (`paper_output_dir`) с
префиксом `suite.output_prefix`. Полный бюджет статьи — `PROFILE='full'`
(гоняет пользователь).